## Cutoff table generator

This notebook computes **cutoff V-values** and the corresponding **cutoff diameters** for guided modes of a circular step-index fiber (core: fused silica via Sellmeier; cladding: air by default).

You provide:
- optical wavelength **λ** (nm)
- maximum fiber diameter **d_max** (µm)

The notebook then scans mode indices *(l, m)* across mode families **TE, TM, HE, EH**, computes each mode's cutoff diameter, and outputs a sorted table of modes whose cutoff diameters are **≤ d_max**.


In [ ]:
# ==== パラメータ入力 ====
λ_nm = 532.0          # Wavelength [nm]
d_max_um = 2.5        # Maximum fiber diameter to consider [µm]
n_clad = 1.000       # Cladding refractive index (air ~ 1.0)

# Scan limits (increase if you expect many modes)
L_MAX = 4            # maximum azimuthal index l to try
M_MAX = 5            # maximum radial index m to try

# Numerical solver settings
ROOT_TOL = 1e-12


In [ ]:
# ==== 計算 ====

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.special as sp
from scipy.optimize import root_scalar
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Sellmeiter方程式を用いた屈折率の計算

def n_fused_silica_sellmeier(λ_nm: float) -> float:
    """Return refractive index of fused silica at wavelength λ_nm using Sellmeier equation.
    λ_nm: wavelength in nm
    """
    λ_um = λ_nm / 1000.0
    λ2 = λ_um ** 2
    n2 = 1.0         + (0.6961663 * λ2) / (λ2 - 0.0684043**2)         + (0.4079426 * λ2) / (λ2 - 0.1162414**2)         + (0.8974794 * λ2) / (λ2 - 9.896161**2)
    return float(np.sqrt(n2))

n_core = n_fused_silica_sellmeier(λ_nm)
NA = np.sqrt(max(n_core**2 - n_clad**2, 0.0))

print(f"λ = {λ_nm:.3f} nm")
print(f"n_core = {n_core:.6f}")
print(f"n_clad = {n_clad:.6f}")

# V値とファイバー径の関係

def V_from_diameter(λ_nm: float, d_um: float, NA: float) -> float:
    """Compute V for given wavelength (nm), diameter (um), and NA."""
    d_nm = d_um * 1e3
    return (np.pi * d_nm / λ_nm) * NA

def dcutoff_from_Vc(λ_nm: float, Vc: float, NA: float) -> float:
    """Cutoff diameter in nm from Vc."""
    if NA == 0:
        return np.inf
    return (Vc * λ_nm) / (np.pi * NA)

V_max = V_from_diameter(λ_nm, d_max_um, NA)
print(f"d_max = {d_max_um:.6f} µm -> V_max = {V_max:.6f}")

# カットオフVcの計算関数

def cutoff_Vc(mode: str, l: int, m: int, n_core: float, n_clad: float) -> float | None:
    """Return cutoff Vc for the requested mode and indices (l,m).
    Returns None if unsupported or if it fails numerically.
    """
    # Special known case: fundamental hybrid mode has no cutoff
    if mode == "HE" and l == 1 and m == 1:
        return 0.0

    try:
        if mode in ("TE", "TM"):
            # Only l=0 is meaningful for TE/TM in this formulation
            if l != 0:
                return None
            # TE0m, TM0m -> m-th zero of J0
            return float(sp.jn_zeros(0, m)[-1])

        if mode == "EH":
            if l <= 0:
                return None
            # EHlm -> m-th zero of J_l
            return float(sp.jn_zeros(l, m)[-1])

        if mode == "HE":
            if l == 1:
                # HE1m: shifted so that HE11 has Vc=0, HE12 uses 1st zero of J1, etc.
                if m <= 1:
                    return 0.0
                return float(sp.jn_zeros(1, m - 1)[-1])

            if l >= 2:
                C = (n_core**2 - n_clad**2) / (n_core**2 + n_clad**2)
                f = lambda v: sp.jn(l - 2, v) + C * sp.jn(l, v)

                # Bracket between consecutive zeros of J_{l-1}
                if m == 1:
                    lo = 1e-3
                    hi = float(sp.jn_zeros(l - 1, 1)[-1])
                else:
                    lo = float(sp.jn_zeros(l - 1, m - 1)[-1])
                    hi = float(sp.jn_zeros(l - 1, m)[-1])

                sol = root_scalar(f, bracket=[lo, hi], xtol=ROOT_TOL, rtol=ROOT_TOL, maxiter=200)
                if sol.converged:
                    return float(sol.root)
                return None

        return None

    except (ValueError, IndexError, RuntimeError):
        return None

# カットオフ径のスキャン関数

def scan_cutoffs(λ_nm: float, d_max_um: float, n_core: float, n_clad: float,
                 L_MAX: int, M_MAX: int) -> pd.DataFrame:
    NA = np.sqrt(max(n_core**2 - n_clad**2, 0.0))
    V_max = V_from_diameter(λ_nm, d_max_um, NA)

    rows = []
    # TE0m and TM0m
    for mode in ("TE", "TM"):
        l = 0
        for m in range(1, M_MAX + 1):
            Vc = cutoff_Vc(mode, l, m, n_core, n_clad)
            if Vc is None:
                continue
            d_nm = dcutoff_from_Vc(λ_nm, Vc, NA)
            if d_nm <= d_max_um * 1e3 + 1e-12:
                rows.append((mode, l, m, Vc, d_nm))
            else:
                # Cutoff diameter grows with m; safe to break for TE/TM
                break

    # HE and EH
    for mode in ("HE", "EH"):
        # l starts at 1 for hybrid modes; HE includes l=1 specially; EH requires l>=1
        for l in range(1, L_MAX + 1):
            # Quick pruning: if even the smallest m=1 cutoff exceeds V_max, higher m will also exceed
            Vc_m1 = cutoff_Vc(mode, l, 1, n_core, n_clad)
            if Vc_m1 is None:
                continue
            if Vc_m1 > V_max + 1e-12 and not (mode == "HE" and l == 1 and 1 == 1):
                # For HE11, Vc=0 so it won't trigger; otherwise prune
                continue

            for m in range(1, M_MAX + 1):
                Vc = cutoff_Vc(mode, l, m, n_core, n_clad)
                if Vc is None:
                    continue
                d_nm = dcutoff_from_Vc(λ_nm, Vc, NA)
                if d_nm <= d_max_um * 1e3 + 1e-12:
                    rows.append((mode, l, m, Vc, d_nm))
                else:
                    # For fixed (mode,l), cutoff increases with m; break m loop
                    break

    df = pd.DataFrame(rows, columns=["mode", "l", "m", "Vc", "d_cutoff_nm"])
    df["d_cutoff_um"] = df["d_cutoff_nm"] / 1e3
    df["mode_label"] = df["mode"] + df["l"].astype(str) + df["m"].astype(str)
    df = df.sort_values(["d_cutoff_nm", "mode", "l", "m"], ascending=[True, True, True, True]).reset_index(drop=True)
    return df

# スキャン実行・表の出力

df = scan_cutoffs(λ_nm, d_max_um, n_core, n_clad, L_MAX=L_MAX, M_MAX=M_MAX)

print(f"Found {len(df)} modes with d_cutoff ≤ {d_max_um} µm")
df.head(50)


In [ ]:
# ==== カットオフ径のプロット ====

import matplotlib.pyplot as plt

if len(df) > 0:
    plt.figure(figsize=(10, 0.11 * min(len(df), 60) + 2))
    y = np.arange(len(df))
    plt.scatter(df["d_cutoff_um"], y)
    plt.yticks(y, df["mode_label"])
    plt.xlabel("Cutoff diameter d_cutoff [µm]")
    plt.ylabel("Mode")
    plt.title(f"Mode cutoff diameters (λ={λ_nm:.0f} nm, d_max={d_max_um:.3f} µm)")
    plt.grid(True, which="both", axis="x")
    plt.tight_layout()
    plt.show()
else:
    print("No modes found within the specified d_max.")


In [ ]:
# ==== CSVファイルの保存 ====
out_csv = f"cutoff_table_lambda{λ_nm:.0f}nm_dmax{d_max_um:.3f}um.csv"
df.to_csv(out_csv, index=False)
out_csv
